# 序列逆置
使用sequence to sequence 模型将一个字符串序列逆置。
例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个sequence to sequence 模型示意图 )
![seq2seq](./seq2seq.png)

In [1]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

I0000 00:00:1775052143.697302 2905254 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775052143.744145 2905254 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775052144.649734 2905254 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [2]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['WAKHHQYOZV', 'QDPJXPOBYS'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[23,  1, 11,  8,  8, 17, 25, 15, 26, 22],
       [17,  4, 16, 10, 24, 16, 15,  2, 25, 19]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0, 22, 26, 15, 25, 17,  8,  8, 11,  1],
       [ 0, 19, 25,  2, 15, 16, 24, 10, 16,  4]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[22, 26, 15, 25, 17,  8,  8, 11,  1, 23],
       [19, 25,  2, 15, 16, 24, 10, 16,  4, 17]], dtype=int32)>)


I0000 00:00:1775052145.787888 2905254 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1032 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:b3:00.0, compute capability: 8.6


# 建立sequence to sequence 模型

In [3]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz = 27
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(128)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(128)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    def call(self, enc_ids, dec_ids):
        # 1.编码器将输入序列压缩为最终隐状态enc_state
        enc_emb = self.embed_layer(enc_ids)
        _, enc_state = self.encoder(enc_emb)
        
        # 2.解码器以enc_state作为初始状态启动
        dec_emb = self.embed_layer(dec_ids)
        dec_out, _ = self.decoder(dec_emb, initial_state=[enc_state])
        
        # 3.映射到词表大小的概率分布
        logits = self.dense(dec_out)
        return logits
    
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) 
        _, enc_state = self.encoder(enc_emb)
        # SimpleRNN只有一个状态，直接返回状态张量即可
        return enc_state
    
    def get_next_token(self, x, state):
        inp_emb = self.embed_layer(x) 
        h, state_list = self.decoder_cell(inp_emb, states=[state]) 
        
        logits = self.dense(h) 
        out = tf.argmax(logits, axis=-1)
        
        # 返回预测的ID和更新后的单一状态
        return tf.cast(out, tf.int32), state_list[0]

# Loss函数以及训练逻辑

In [4]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(3000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [5]:
optimizer = optimizers.Adam(0.001)
model = mySeq2SeqModel()

# 注入Dummy Batch提前构建计算图，避免GradientTape抓取不到变量
_, dummy_enc_x, dummy_dec_x, _ = get_batch(2, 20)
model(dummy_enc_x, dummy_dec_x)
print("模型权重已就绪，开始训练基础版Seq2Seq...")

train(model, optimizer, seqlen=20)

模型权重已就绪，开始训练基础版Seq2Seq...
step 0 : loss 3.2949424
step 500 : loss 1.2301369
step 1000 : loss 0.75585717
step 1500 : loss 0.546018
step 2000 : loss 0.4332622
step 2500 : loss 0.30063796


<tf.Tensor: shape=(), dtype=float32, numpy=0.2797902226448059>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [6]:
def sequence_reversal():
    def decode(init_state, steps=10):
        b_sz = tf.shape(init_state)[0] 
        
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state)
            collect.append(tf.expand_dims(cur_token, axis=-1))
            
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 10)
    state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1]), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    return seq == rev_seq_rev

print("\n---正在测试模型逆置能力---")
results = list(zip(*sequence_reversal()))
print("准确率判断:", [is_reverse(*item) for item in results])

print("\n实际结果比对(预测结果, 原始输入):")
for r in results[:5]:
    print(r)


---正在测试模型逆置能力---
准确率判断: [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]

实际结果比对(预测结果, 原始输入):
('RZHQTIMXQT', 'TQXMITQHZR')
('TWXSEEMBSH', 'HSBMEESXWT')
('RBOYMDWARK', 'KRAWDMYOBR')
('CVHWZCPIHR', 'RHIPCZWHVC')
('LQKDHCHEOO', 'OOEHCHDKQL')
